<div align="center">

# NB3 · The XAI Toolkit II: Medical Imaging

**Explainable Artificial Intelligence in Medical Systems: From Black Box Models to Reliable Clinical Decision Support**<br>
IEEE CARS 2026 Workshop · Grand Forks, ND, USA · 28 October 2026 · Session 10:20 to 10:45

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/utkukose/xai-cars2026-NB-workshop/blob/main/notebooks/NB3_XAI_Toolkit_Imaging.ipynb)

Prof. Dr. Utku Köse · Süleyman Demirel University · University of North Dakota · Universidad Panamericana · Vel Tech University<br>
[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

</div>

## About this notebook

Heat maps over chest radiographs are the most familiar picture of explainable AI, and also the easiest to over-trust. This notebook trains a small convolutional network to detect pneumonia on real paediatric chest X-rays, explains its decisions with five saliency methods, and then tests the explanations. The sanity check of Adebayo et al. [1] randomises the network layer by layer: an explanation that barely changes when the model is destroyed was never explaining the model. A deletion test measures faithfulness, and a final section shows that a change invisible to the eye can flip both the prediction and its heat map.

The data are PneumoniaMNIST from MedMNIST v2 [2], derived from the paediatric chest radiographs of Kermany et al. [3] and distributed under CC BY 4.0. The 64 by 64 pixel version (about 20 MB) is downloaded from Zenodo when the notebook runs. On a free Colab CPU the full run takes three to five minutes; with a GPU runtime it takes about one minute.

In [ ]:
import os, io, sys, json, time, zipfile, hashlib, warnings, urllib.request, urllib.error
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

QUICK = os.environ.get("XAI_WORKSHOP_QUICK", "0") == "1"   # smaller runs for automated checks
SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("workshop_data")
DATA_DIR.mkdir(exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
trapezoid = getattr(np, "trapezoid", None) or np.trapz

INK, ALERT, SAFE, MUTED, AMBER, VIOLET = "#1D2742", "#C0392B", "#1F7A6D", "#8A93A6", "#B26B00", "#6C4AB6"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
    "text.color": INK, "axes.titleweight": "bold", "axes.titlesize": 11,
})


def fetch(urls, dest, timeout=120):
    """Download the first reachable URL into dest, reuse an existing copy, and return the path."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    errors = []
    for url in ([urls] if isinstance(urls, str) else urls):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                payload = response.read()
            part = dest.with_name(dest.name + ".part")
            part.write_bytes(payload)
            part.replace(dest)
            return dest
        except Exception as exc:
            errors.append(f"{url}: {exc}")
    raise RuntimeError("No source could be reached:\n" + "\n".join(errors))


print(f"numpy {np.__version__} | pandas {pd.__version__} | quick mode: {QUICK}")

In [ ]:
import importlib.util
if importlib.util.find_spec("torch") is None:
    raise ModuleNotFoundError("PyTorch is needed for this notebook. It is preinstalled on Colab; locally run "
                              "pip install torch --index-url https://download.pytorch.org/whl/cpu")
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import ipywidgets as widgets
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)

## 1. Data

In [ ]:
NPZ_URL = "https://zenodo.org/records/10519652/files/pneumoniamnist_64.npz?download=1"
NPZ_MD5 = "8f4eceb4ccffa70c672198ea285246c6"
npz_path = fetch([NPZ_URL], DATA_DIR / "pneumoniamnist_64.npz", timeout=300)
if hashlib.md5(npz_path.read_bytes()).hexdigest() != NPZ_MD5:
    print("Note: the file checksum differs from the published MedMNIST value.")
arrays = np.load(npz_path)
X_tr, y_tr = arrays["train_images"], arrays["train_labels"].ravel().astype(int)
X_va, y_va = arrays["val_images"], arrays["val_labels"].ravel().astype(int)
X_te, y_te = arrays["test_images"], arrays["test_labels"].ravel().astype(int)
if QUICK:
    X_tr, y_tr = X_tr[:1200], y_tr[:1200]
CLASSES = ["normal", "pneumonia"]
for name, lab in [("train", y_tr), ("validation", y_va), ("test", y_te)]:
    print(f"{name:>10}: {len(lab):5d} images · {lab.mean():.0%} pneumonia")

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for row, cls in enumerate([0, 1]):
    for col, i in enumerate(np.where(y_tr == cls)[0][:6]):
        axes[row, col].imshow(X_tr[i], cmap="gray")
        axes[row, col].set_title(CLASSES[cls], fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()
plt.show()

## 2. A small convolutional network

Four convolutional blocks reduce the 64 by 64 image to an 8 by 8 map of 64 channels, which is averaged and passed to a linear classifier. Class weights compensate for the higher share of pneumonia images. Training runs for a few epochs; the goal is a reasonable model to explain, not a state-of-the-art one.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()

        def block(c_in, c_out):
            return [nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2)]

        self.features = nn.Sequential(*block(1, 16), *block(16, 32), *block(32, 64),
                                      nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, 2)

    def forward(self, x):
        return self.fc(self.pool(self.features(x)).flatten(1))


def to_tensor(images):
    return torch.tensor(np.asarray(images)[:, None] / 255.0, dtype=torch.float32)


def train(model, X, y, epochs, batch=64, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    counts = np.bincount(y, minlength=2)
    weight = torch.tensor(len(y) / (2.0 * counts), dtype=torch.float32, device=DEVICE)
    Xt, yt = to_tensor(X).to(DEVICE), torch.tensor(y, device=DEVICE)
    for epoch in range(epochs):
        model.train()
        perm, total = torch.randperm(len(Xt), device=DEVICE), 0.0
        for b in range(0, len(Xt), batch):
            idx = perm[b:b + batch]
            loss = F.cross_entropy(model(Xt[idx]), yt[idx], weight=weight)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        print(f"epoch {epoch + 1}: training loss {total / len(Xt):.3f}")
    return model.eval()


@torch.no_grad()
def predict_proba(model, images, batch=256):
    model.eval()
    return np.concatenate([F.softmax(model(to_tensor(images[b:b + batch]).to(DEVICE)), dim=1)[:, 1].cpu().numpy()
                           for b in range(0, len(images), batch)])


model = train(SmallCNN().to(DEVICE), X_tr, y_tr, epochs=2 if QUICK else 6)
p_te = predict_proba(model, X_te)
pred_te = (p_te >= 0.5).astype(int)
sens = (pred_te[y_te == 1] == 1).mean()
spec = (pred_te[y_te == 0] == 0).mean()
print(f"test AUROC {roc_auc_score(y_te, p_te):.3f} · sensitivity {sens:.3f} · specificity {spec:.3f}")

## 3. Five ways to draw a heat map

**Grad-CAM** weights the last convolutional feature maps by the average gradient of the class score and keeps the positive part [4]. **Vanilla gradients** show how sensitive the class score is to each pixel [5]. **SmoothGrad** averages gradients over noisy copies of the image to reduce visual noise [6]. **Integrated gradients** accumulate gradients along a path from a black image to the input and satisfy completeness [7]. **Occlusion** covers the image patch by patch with the mean grey value and records the drop in the predicted probability [8]. All maps below explain the pneumonia class.

In [ ]:
def image_tensor(i, images=X_te):
    return to_tensor(images[i:i + 1]).to(DEVICE)


def grad_cam(net, x, target=1):
    feats = net.features(x)
    logits = net.fc(net.pool(feats).flatten(1))
    grads, = torch.autograd.grad(logits[0, target], feats)
    cam = F.relu((grads.mean(dim=(2, 3), keepdim=True) * feats).sum(dim=1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    return (cam / (cam.max() + 1e-8)).detach().cpu().numpy()


def vanilla_gradient(net, x, target=1):
    x = x.clone().requires_grad_(True)
    g, = torch.autograd.grad(net(x)[0, target], x)
    return g[0, 0].abs().detach().cpu().numpy()


def smoothgrad(net, x, target=1, n=25, sigma=0.1):
    noisy = (x.repeat(n, 1, 1, 1) + sigma * torch.randn(n, *x.shape[1:], device=x.device)).requires_grad_(True)
    g, = torch.autograd.grad(net(noisy)[:, target].sum(), noisy)
    return g.mean(dim=0)[0].abs().detach().cpu().numpy()


def integrated_gradients(net, x, target=1, steps=32):
    base = torch.zeros_like(x)
    alphas = ((torch.arange(steps, device=x.device) + 0.5) / steps).view(-1, 1, 1, 1)
    path = (base + alphas * (x - base)).requires_grad_(True)
    g, = torch.autograd.grad(net(path)[:, target].sum(), path)
    return ((x - base) * g.mean(dim=0, keepdim=True))[0, 0].detach().cpu().numpy()


@torch.no_grad()
def occlusion(net, x, target=1, patch=8, stride=4):
    base = F.softmax(net(x), dim=1)[0, target].item()
    fill, size = x.mean().item(), x.shape[-1]
    coords = [(i, j) for i in range(0, size - patch + 1, stride) for j in range(0, size - patch + 1, stride)]
    batch = x.repeat(len(coords), 1, 1, 1)
    for k, (i, j) in enumerate(coords):
        batch[k, :, i:i + patch, j:j + patch] = fill
    drop = base - F.softmax(net(batch), dim=1)[:, target].cpu().numpy()
    heat, count = np.zeros((size, size)), np.zeros((size, size))
    for (i, j), d in zip(coords, drop):
        heat[i:i + patch, j:j + patch] += d
        count[i:i + patch, j:j + patch] += 1
    return heat / np.maximum(count, 1)


METHODS = {"Grad-CAM": grad_cam, "Gradient": vanilla_gradient, "SmoothGrad": smoothgrad,
           "Integrated gradients": integrated_gradients, "Occlusion": occlusion}

In [ ]:
groups = [("true positive", (y_te == 1) & (pred_te == 1)), ("missed pneumonia", (y_te == 1) & (pred_te == 0)),
          ("true negative", (y_te == 0) & (pred_te == 0)), ("false alarm", (y_te == 0) & (pred_te == 1))]
examples = [(label, int(np.where(mask)[0][0])) for label, mask in groups if mask.any()][:3]
tp = np.where((y_te == 1) & (pred_te == 1))[0]
if len(tp) == 0:                      # fall back to all pneumonia images if none is detected
    tp = np.where(y_te == 1)[0]

fig, axes = plt.subplots(len(examples), len(METHODS) + 1, figsize=(15, 2.8 * len(examples)))
for r, (label, i) in enumerate(examples):
    x = image_tensor(i)
    axes[r, 0].imshow(X_te[i], cmap="gray")
    axes[r, 0].set_title(f"{label}\nP(pneumonia) = {p_te[i]:.2f}", fontsize=9)
    for c, (name, fn_map) in enumerate(METHODS.items(), start=1):
        m = fn_map(model, x)
        axes[r, c].imshow(X_te[i], cmap="gray")
        axes[r, c].imshow(np.clip(m, 0, None), cmap="inferno", alpha=0.55)
        axes[r, c].set_title(name, fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

x0 = image_tensor(tp[0])
ig_map = integrated_gradients(model, x0, steps=128)
with torch.no_grad():
    delta = (model(x0)[0, 1] - model(torch.zeros_like(x0))[0, 1]).item()
print(f"integrated gradients completeness: sum {ig_map.sum():+.3f} vs logit difference {delta:+.3f}")

## 4. Sanity check: does the heat map depend on the model?

The cascading randomisation test re-initialises the network from the top: first the classifier, then the convolutional layers one by one down to the input [1]. A trustworthy explanation should change markedly as soon as the learned weights are destroyed. The plot reports the rank correlation between each map and the map of the trained model, averaged over several test images. A method whose curve stays near 1 is showing the image, not the model.

In [ ]:
conv_layers = [m for m in model.features if isinstance(m, nn.Conv2d)]
stages = [("trained", [])] + [("classifier", ["fc"])] + [
    (f"+ conv {k}", ["fc"] + [f"conv{j}" for j in range(len(conv_layers), k - 1, -1)])
    for k in range(len(conv_layers), 0, -1)]


def randomised_copy(names):
    net = copy.deepcopy(model)
    convs = [m for m in net.features if isinstance(m, nn.Conv2d)]
    torch.manual_seed(SEED)
    for name in names:
        layer = net.fc if name == "fc" else convs[int(name[4:]) - 1]
        layer.reset_parameters()
    return net.eval()


def rank_corr(a, b):
    a, b = np.abs(a).ravel(), np.abs(b).ravel()
    return 0.0 if a.std() == 0 or b.std() == 0 else spearmanr(a, b)[0]   # a flat map carries no information


probe = list(tp[:4 if QUICK else 8])
reference = {name: [f(model, image_tensor(i)) for i in probe] for name, f in METHODS.items()}
curves = {name: [] for name in METHODS}
for stage, names in stages:
    net = randomised_copy(names)
    for name, f in METHODS.items():
        rho = [rank_corr(f(net, image_tensor(i)), ref) for i, ref in zip(probe, reference[name])]
        curves[name].append(np.mean(rho))

fig, ax = plt.subplots(figsize=(8, 3.8))
for (name, vals), col in zip(curves.items(), [ALERT, INK, VIOLET, SAFE, AMBER]):
    ax.plot([s for s, _ in stages], vals, "o-", color=col, label=name)
ax.set_ylabel("rank correlation with trained-model map")
ax.set_ylim(-0.2, 1.05)
ax.legend(frameon=False, fontsize=8)
ax.set_title("Cascading randomisation: lower means the map depends on the learned weights")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 5. Faithfulness: deleting what the map calls important

The deletion test removes the image in 4 by 4 pixel patches, most important first, by filling them with the mean grey value, and follows the predicted probability of pneumonia [9]. A faithful map produces a steep early drop and therefore a small area under the curve. A random ordering provides the reference.

In [ ]:
def deletion_curve(net, x, attribution, steps=16, patch=4):
    size = x.shape[-1]
    per_side = size // patch
    score = attribution.reshape(per_side, patch, per_side, patch).mean(axis=(1, 3)).ravel()
    order = np.argsort(-score)
    per_step = len(order) // steps
    cur, fill, frames = x.clone(), x.mean(), [x.clone()]
    for s in range(steps):
        for k in order[s * per_step:(s + 1) * per_step]:
            r, c = divmod(int(k), per_side)
            cur[..., r * patch:(r + 1) * patch, c * patch:(c + 1) * patch] = fill
        frames.append(cur.clone())
    with torch.no_grad():
        return F.softmax(net(torch.cat(frames)), dim=1)[:, 1].cpu().numpy()


def signed_for_deletion(name, m):
    return np.clip(m, 0, None) if name in ("Integrated gradients", "Occlusion") else m


probe_del = list(tp[:6 if QUICK else 16])
del_curves = {name: [] for name in list(METHODS) + ["random"]}
for i in probe_del:
    x = image_tensor(i)
    for name, f in METHODS.items():
        del_curves[name].append(deletion_curve(model, x, signed_for_deletion(name, f(model, x))))
    del_curves["random"].append(deletion_curve(model, x, rng.random((64, 64))))

fig, ax = plt.subplots(figsize=(7.5, 3.8))
steps_axis = np.linspace(0, 1, 17)
areas = {}
for (name, cs), col in zip(del_curves.items(), [ALERT, INK, VIOLET, SAFE, AMBER, MUTED]):
    mean_curve = np.mean(cs, axis=0)
    areas[name] = trapezoid(mean_curve, steps_axis)
    ax.plot(steps_axis, mean_curve, color=col, lw=2 if name != "random" else 1.5,
            ls="-" if name != "random" else "--", label=f"{name} (area {areas[name]:.3f})")
ax.set_xlabel("fraction of the image removed")
ax.set_ylabel("P(pneumonia)")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## 6. A preview of Part 3: the explanation can be attacked too

The fast gradient sign method (FGSM) adds a perturbation of fixed size in the direction that increases the model's loss [10]. With a budget of a few grey levels out of 255 the change is invisible, yet predictions flip. This is a white-box attack that uses the model's gradients; the attacks in NB4 to NB6 work with far less access. Heat maps react as well, which is why explanations themselves can be manipulated [11].

In [ ]:
def fgsm(net, x, label, eps):
    x = x.clone().requires_grad_(True)
    loss = F.cross_entropy(net(x), torch.tensor([label], device=x.device))
    g, = torch.autograd.grad(loss, x)
    return (x + eps * g.sign()).clamp(0, 1).detach()


subset = np.arange(min(len(y_te), 120 if QUICK else 400))
rows = []
for eps in [0, 1 / 255, 2 / 255, 4 / 255]:
    adv = torch.cat([fgsm(model, image_tensor(i), int(y_te[i]), eps) for i in subset])
    with torch.no_grad():
        p_adv = F.softmax(model(adv), dim=1)[:, 1].cpu().numpy()
    rows.append({"epsilon (grey levels)": round(eps * 255), "accuracy": ((p_adv >= 0.5) == y_te[subset]).mean()})
pd.DataFrame(rows).round(3)

In [ ]:
flipped = next((i for i in tp if (F.softmax(model(fgsm(model, image_tensor(i), 1, 2 / 255)), 1)[0, 1] < 0.5).item()),
               tp[0])
x = image_tensor(flipped)
x_adv = fgsm(model, x, 1, 2 / 255)
with torch.no_grad():
    p_clean, p_adv = (F.softmax(model(t), 1)[0, 1].item() for t in (x, x_adv))
panels = [(x[0, 0].cpu(), f"original, P = {p_clean:.2f}", "gray"),
          (x_adv[0, 0].cpu(), f"manipulated, P = {p_adv:.2f}", "gray"),
          ((x_adv - x)[0, 0].cpu() * 40, "difference x 40", "RdBu"),
          (grad_cam(model, x), "Grad-CAM, original", "inferno"),
          (grad_cam(model, x_adv), "Grad-CAM, manipulated", "inferno")]
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, (img, title, cmap) in zip(axes, panels):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Explore

Choose a test image and a method. The title shows the true label and the model's probability.

In [ ]:
def explore(index=0, method="Grad-CAM"):
    i = int(index)
    m = METHODS[method](model, image_tensor(i))
    fig, axes = plt.subplots(1, 2, figsize=(7, 3.4))
    axes[0].imshow(X_te[i], cmap="gray")
    axes[1].imshow(X_te[i], cmap="gray")
    axes[1].imshow(np.clip(m, 0, None), cmap="inferno", alpha=0.55)
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"test image {i}: {CLASSES[y_te[i]]}, P(pneumonia) = {p_te[i]:.2f}")
    plt.tight_layout()
    plt.show()


widgets.interact(explore, index=widgets.IntSlider(value=0, min=0, max=len(X_te) - 1, continuous_update=False),
                 method=widgets.Dropdown(options=list(METHODS)));

## Discussion

1. Which of the five methods would you show to a radiologist, and which test result supports that choice?
2. Grad-CAM maps are coarse because they come from an 8 by 8 layer. When is coarse acceptable in clinical use?
3. If a heat map can change without any visible change to the image, what should a hospital log alongside each explanation?

## Summary

Saliency methods differ in what they measure, and their visual appeal is not evidence of faithfulness. The randomisation test separates maps that depend on the learned model from maps that mostly reflect image structure, and the deletion test ranks methods by how well their ranking predicts the model's behaviour. Both tests take a few lines of code and should accompany any heat map shown in a clinical setting. Part 3 of the workshop turns to deliberate attacks on medical AI.

## References

[1] J. Adebayo, J. Gilmer, M. Muelly, I. Goodfellow, M. Hardt, and B. Kim, "Sanity checks for saliency maps," in *Advances in Neural Information Processing Systems 31*, 2018. arXiv:1810.03292

[2] J. Yang et al., "MedMNIST v2: A large-scale lightweight benchmark for 2D and 3D biomedical image classification," *Scientific Data*, vol. 10, article 41, 2023. doi: 10.1038/s41597-022-01721-8

[3] D. S. Kermany et al., "Identifying medical diagnoses and treatable diseases by image-based deep learning," *Cell*, vol. 172, no. 5, pp. 1122–1131.e9, 2018. doi: 10.1016/j.cell.2018.02.010

[4] R. R. Selvaraju, M. Cogswell, A. Das, R. Vedantam, D. Parikh, and D. Batra, "Grad-CAM: Visual explanations from deep networks via gradient-based localization," in *Proc. IEEE Int. Conf. Computer Vision (ICCV)*, 2017, pp. 618–626. doi: 10.1109/ICCV.2017.74

[5] K. Simonyan, A. Vedaldi, and A. Zisserman, "Deep inside convolutional networks: Visualising image classification models and saliency maps," ICLR Workshop, 2014. arXiv:1312.6034

[6] D. Smilkov, N. Thorat, B. Kim, F. Viégas, and M. Wattenberg, "SmoothGrad: Removing noise by adding noise," arXiv:1706.03825, 2017.

[7] M. Sundararajan, A. Taly, and Q. Yan, "Axiomatic attribution for deep networks," in *Proc. 34th Int. Conf. Machine Learning (ICML)*, PMLR 70, 2017, pp. 3319–3328. arXiv:1703.01365

[8] M. D. Zeiler and R. Fergus, "Visualizing and understanding convolutional networks," in *Computer Vision, ECCV 2014*, LNCS 8689, 2014, pp. 818–833. doi: 10.1007/978-3-319-10590-1_53

[9] W. Samek, A. Binder, G. Montavon, S. Lapuschkin, and K.-R. Müller, "Evaluating the visualization of what a deep neural network has learned," *IEEE Transactions on Neural Networks and Learning Systems*, vol. 28, no. 11, pp. 2660–2673, 2017. doi: 10.1109/TNNLS.2016.2599820

[10] I. J. Goodfellow, J. Shlens, and C. Szegedy, "Explaining and harnessing adversarial examples," in *Proc. Int. Conf. Learning Representations (ICLR)*, 2015. arXiv:1412.6572

[11] A. Ghorbani, A. Abid, and J. Zou, "Interpretation of neural networks is fragile," in *Proc. 33rd AAAI Conf. Artificial Intelligence*, 2019. arXiv:1710.10547